# XFdtd volume field map: qualification and RF-Track artifact

This notebook is the reviewable front end for the new six-component XFdtd HDF5 workflow. It inspects the 24.8 GB source lazily, plans bounded out-of-core reads, fits complex E/H phasors, measures cylindrical symmetry, applies the declared vector-$m=0$ projection, qualifies the downstream tail, and writes a digest-verified compact artifact. It has no RF-Track dependency.

The raw export contains electric field **E** in V/m and magnetic-field intensity **H** in A/m. The reduction converts $\mathbf B=\mu_0\mathbf H$ after coordinate transformation and only in the physical vacuum mask. The production tracking representation remains axisymmetric even when the diagnostics show that B is less axisymmetric than E.
Run top to bottom in a fresh kernel. The default **review mode** reuses the compact artifact; `RUN_REDUCTION=True` reads the large source and may take tens of minutes. Writing and overwriting are separate explicit settings. Select the fill policy before rebuilding; the current step-drive correction is provisional ([README](README.md#field-maps-from-xfdtd)).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import rf_gun.fieldmaps as fm
from rf_gun.plotting import field_analysis as field_plots
from rf_gun.fieldmaps.comparison import compare_tail_control
from rf_gun.plotting.figure_io import save_figure_formats

SOURCE = Path("field_maps/cavity_E_H_full_volume.h5")
ARTIFACT = Path("field_maps/cavity_axisymmetric_m0_rftrack.h5")
ZERO_CONTROL = Path("field_maps/cavity_axisymmetric_m0_zero_tail_control.h5")
EXPECTED_SOURCE_SHA256 = fm.CAVITY_SOURCE_SHA256

TARGET_Z_MAX_M = 40.589e-3
DR_M = 100.0e-6
DZ_M = 50.0e-6
THETA_COUNT = 32
TAIL_FIT_START_M = 30.0e-3
TAIL_POLICY = "modal"       # "modal" for production; "zero" is the control

FILL_POLICY = "auto"           # "single-pole" is a provisional step-drive model; "off" is raw
FILL_START_TIME_NS = None      # explicitly set 0.0 only when assuming a step at t=0

RUN_REDUCTION = False          # Deliberately safe: the source is 24.8 GB
WRITE_ARTIFACT = False        # enable only to save a new reduction
WRITE_ZERO_CONTROL = True    # write its matching zero-tail control with the artifact
OVERWRITE_ARTIFACT = False
VERIFY_SOURCE_SHA256 = False   # True streams through all 24.8 GB
SOURCE_DIGEST_PREVIOUSLY_CONFIRMED = True

FIGURES_DIR = Path("outputs/field_map_qualification/figures")
SAVE_FIGURES = False  # enable to save diagnostics to FIGURES_DIR

print(f"source:   {SOURCE}")
print(f"artifact: {ARTIFACT}")
print(f"control:  {ZERO_CONTROL}")
print(f"figures:  {FIGURES_DIR} (SAVE_FIGURES={SAVE_FIGURES})")
print(f"RUN_REDUCTION={RUN_REDUCTION}, WRITE_ARTIFACT={WRITE_ARTIFACT}")


## 1. Lazy source inspection and schema validation

`inspect_xfdtd_h5` reads attributes, short coordinate arrays, and the two 17-element time arrays; it does not read any 4-D field payload. `validate_xfdtd_h5` likewise checks schema, dimensions, units, coordinate monotonicity, and timestamps without loading the volume. This interactive notebook leaves full-file hashing opt-in after the source digest has already been confirmed; the routine CLI verifies all 24.8 GB by default. A separate bounded check reads only one all-frame normal-electric-field line to verify that the declared origin is physically the cathode surface.

In [ ]:
if not SOURCE.exists():
    raise FileNotFoundError(SOURCE)

manifest = fm.inspect_xfdtd_h5(SOURCE)
source_report = fm.validate_xfdtd_h5(
    SOURCE,
    expected_sha256=EXPECTED_SOURCE_SHA256 if VERIFY_SOURCE_SHA256 else None,
)
source_report.raise_for_errors()

display({
    "size_GB": manifest.file_size_bytes / 1.0e9,
    "schema_version": manifest.schema_version,
    "array_axes": manifest.array_axes,
    "solver": manifest.root_attributes.get("solver"),
    "source_run_id": manifest.root_attributes.get("source_run_id"),
    "drive_frequency_GHz": manifest.drive_frequency_hz / 1.0e9,
    "source_power_MW": manifest.source_power_w / 1.0e6,
    "cathode_origin_native_mm": 1.0e3 * manifest.cathode_origin_native_m,
    "validation_warnings": [issue.message for issue in source_report.warnings],
})

display([{
    "component": name,
    "shape_(t,x,y,z)": item.shape,
    "chunks": item.chunks,
    "dtype": item.dtype,
    "units": item.units,
} for name, item in manifest.components.items()])

print("native -> beam rotation:")
display(fm.XFD_TD_CATHODE_TO_BEAM.R_target_from_native)
print("(x_beam, y_beam, z_beam) = (X_native, Z_native, -Y_native)")

cathode_origin_check = fm.check_xfdtd_cathode_origin(SOURCE)
cathode_origin_report = cathode_origin_check.as_dict()
display(cathode_origin_report)
if not cathode_origin_check.origin_physically_bracketed:
    raise RuntimeError("The normal-E transition does not physically bracket beam z=0")

origin_window = np.abs(cathode_origin_check.beam_z_m) <= 0.6e-3
origin_amplitude = np.abs(cathode_origin_check.normal_field_phasor_Vpm)
fig, ax = plt.subplots(figsize=(8.5, 4.0), constrained_layout=True)
ax.semilogy(
    cathode_origin_check.beam_z_m[origin_window] * 1.0e6,
    np.maximum(origin_amplitude[origin_window], np.finfo(float).tiny),
    "o-", label=r"$|\hat E_z|$ on axis",
)
ax.axvline(0.0, color="black", ls="--", label="declared cathode origin")
ax.axvspan(
    cathode_origin_check.material_sample_beam_z_m * 1.0e6,
    cathode_origin_check.vacuum_sample_beam_z_m * 1.0e6,
    color="C1", alpha=0.2, label="field-transition bracket",
)
ax.set(
    xlabel=r"beam $z$ relative to declared origin ($\mu$m)",
    ylabel=r"normal-field phasor amplitude (V/m)",
    title="Physical cathode-origin verification from the bounded XFdtd field line",
)
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=False)
plt.show()

In [ ]:
coordinate_axes_m = {}
with fm.XFdtdH5(SOURCE) as reader:
    for quantity in ("E", "H"):
        for component in "xyz":
            component_axes = reader.coordinates(quantity, component, frame="cathode_centered")
            for axis_name, axis_values in zip("xyz", component_axes, strict=True):
                coordinate_axes_m[f"{quantity}{component}:{axis_name}"] = axis_values

period_s = 1.0 / manifest.drive_frequency_hz
display({
    "E_frame_count": manifest.electric_time_s.size,
    "H_frame_count": manifest.magnetic_h_time_s.size,
    "E_mean_cadence_ps": 1.0e12 * np.mean(np.diff(manifest.electric_time_s)),
    "H_mean_cadence_ps": 1.0e12 * np.mean(np.diff(manifest.magnetic_h_time_s)),
    "E_samples_per_RF_cycle": period_s / np.mean(np.diff(manifest.electric_time_s)),
    "H_samples_per_RF_cycle": period_s / np.mean(np.diff(manifest.magnetic_h_time_s)),
    "E_minus_H_first_timestamp_ps": 1.0e12 * (manifest.electric_time_s[0] - manifest.magnetic_h_time_s[0]),
})
field_plots.plot_mesh_time_metadata(
    coordinate_axes_m,
    manifest.electric_time_s,
    manifest.magnetic_h_time_s,
    manifest.drive_frequency_hz,
    show=True,
)

The E and H samples have separate saved-time arrays and every Cartesian component has its own Yee-staggered coordinates. The reducer therefore fits and interpolates each component independently; it never assigns one component's mesh or time base to another. The coordinate metadata place the cathode at the declared origin, while the bounded normal-$E$ check independently confirms a material-to-vacuum transition that brackets $z=0$ at the native mesh resolution. The beam-axis orientation is fixed by the proper rotation shown above.

## 2. Common support, RF grid, and vacuum mask

The planned measured prefix stays inside the common support of all six components and all 32 azimuths. The analytic aperture excludes metal during reduction and diagnostics. The same uniform $z$ step continues to the tracking-domain endpoint so a tail policy cannot introduce a grid discontinuity.

In [ ]:
grid_plan = fm.plan_axisymmetric_grid(
    SOURCE,
    target_z_max_m=TARGET_Z_MAX_M,
    requested_dr_m=DR_M,
    requested_dz_m=DZ_M,
    wall_margin_m=0.0,
)
display(grid_plan.as_metadata())

fig, ax = plt.subplots(figsize=(10.5, 4.0), constrained_layout=True)
ax.pcolormesh(
    grid_plan.full_z_m * 1.0e3,
    grid_plan.r_m * 1.0e3,
    grid_plan.full_vacuum_mask.T.astype(float),
    shading="auto", cmap="Greys_r", vmin=0.0, vmax=1.0,
)
ax.plot(grid_plan.full_z_m * 1.0e3, grid_plan.full_aperture_radius_m * 1.0e3, "C1", label="analytic aperture")
ax.axvline(grid_plan.source_support.z_max_m * 1.0e3, color="C3", ls="--", label="common H5 support end")
ax.axvline(grid_plan.measured_z_max_m * 1.0e3, color="C0", ls=":", label="last aligned measured sample")
ax.set(xlabel="beam z (mm)", ylabel="radius (mm)", title="Planned vacuum grid: white is vacuum, black is excluded material")
ax.legend(frameon=False)
if SAVE_FIGURES:
    print(save_figure_formats(fig, FIGURES_DIR, "planned_vacuum_grid", formats=["png"]))
plt.show()

In [ ]:
theta_rad = np.linspace(0.0, 2.0 * np.pi, THETA_COUNT, endpoint=False)
harmonic_config = fm.HarmonicFitConfig(
    manifest.drive_frequency_hz,
    include_offset=True,
    include_linear_envelope=True,
)
reduction_config = fm.ReductionConfig(
    r_m=grid_plan.r_m,
    z_m=grid_plan.measured_z_m,
    theta_rad=theta_rad,
    vacuum_mask=grid_plan.measured_vacuum_mask,
    harmonic=harmonic_config,
    longitudinal_block_points=64,
    interpolation_guard_cells=1,
    wall_guard_cells=2,
    beam_core_radius_m=2.0e-3,
    m_max=4,
    output_dtype="complex64",
    max_logical_read_bytes=512 * 1024 * 1024,
)
display(reduction_config.as_metadata())

## 3. Bounded volume reduction

This is the only expensive cell. For every bounded longitudinal slab it reads one component/frame selection, performs the fixed-frequency all-frame fit, co-locates the staggered components on cylindrical samples, transforms vectors, converts H to B in vacuum, records azimuthal modes, and retains the cylindrical vector-$m=0$ fields. Leave `RUN_REDUCTION=False` when reviewing an existing artifact.

In [ ]:
reduction_result = None
if WRITE_ARTIFACT and not RUN_REDUCTION:
    raise ValueError("Writing requires RUN_REDUCTION=True")
if RUN_REDUCTION and WRITE_ARTIFACT:
    destinations = [ARTIFACT] + ([ZERO_CONTROL] if WRITE_ZERO_CONTROL else [])
    resolved = [SOURCE.resolve(), *(path.resolve() for path in destinations)]
    if len(resolved) != len(set(resolved)):
        raise ValueError("Source, artifact and control paths must be distinct")
    if not OVERWRITE_ARTIFACT and any(path.exists() for path in destinations):
        raise FileExistsError("Choose new output paths or explicitly enable OVERWRITE_ARTIFACT before reducing")
if RUN_REDUCTION:
    if FILL_POLICY == "single-pole" and FILL_START_TIME_NS is None:
        raise ValueError("Set FILL_START_TIME_NS to declare the step-drive assumption")
    reduction_result = fm.reduce_xfdtd_to_axisymmetric(
        SOURCE, reduction_config, progress=None
    )
    print("bounded reduction complete")
else:
    print("Reduction skipped. Set RUN_REDUCTION=True only for an intentional rebuild.")


In [ ]:
loaded_artifact = None
if not RUN_REDUCTION and ARTIFACT.exists():
    loaded_artifact = fm.read_axisymmetric_artifact(
        ARTIFACT, verify="payload", require_qualified=False
    )
    print(f"loaded and payload-verified: {ARTIFACT}")
    print(f"payload SHA-256: {loaded_artifact.payload_sha256}")
    print(f"quality status: {loaded_artifact.quality.get('status')}")
elif not RUN_REDUCTION:
    print("No compact artifact exists yet. Run prepare_xfdtd_fieldmap.py or enable the reduction once.")

## 4. Temporal fit and cylindrical-symmetry evidence

Non-axisymmetric content is measured before projection. It is a diagnostic of the source solution, not a hidden hard failure. In particular, the magnetic field is expected to be noticeably less axisymmetric. The declared production choice remains the vector $m=0$ projection for RF-Track.

In [ ]:
if reduction_result is not None:
    reduction_report = reduction_result.report.as_dict()
elif loaded_artifact is not None:
    reduction_report = dict(
        loaded_artifact.quality.get("field_reduction",
        loaded_artifact.quality.get("reduction", {}))
    )
else:
    reduction_report = {}

temporal_diagnostics = reduction_report.get("temporal", {})
symmetry_diagnostics = reduction_report.get("symmetry", {})
if temporal_diagnostics:
    display({
        "production_metrics": temporal_diagnostics.get("production_metrics", {}),
        "E_H_family_aggregate": temporal_diagnostics.get("family_aggregate", {}),
        "E_H_family_block_aggregate": temporal_diagnostics.get("family_block_aggregate", {}),
        "unfiltered_worst_case_diagnostics": {key: temporal_diagnostics[key] for key in (
            "max_global_nrmse", "max_active_nrmse_p95",
            "max_peak_fractional_drift_per_cycle") if key in temporal_diagnostics},
    })
    field_plots.plot_temporal_fit_drift(temporal_diagnostics, show=True)
else:
    print("Temporal diagnostics will appear after reduction or artifact loading.")

if symmetry_diagnostics:
    e_nonaxis = float(symmetry_diagnostics["combined_e_nonaxisymmetric_fraction"])
    b_nonaxis = float(symmetry_diagnostics["combined_b_nonaxisymmetric_fraction"])
    _core_symmetry = symmetry_diagnostics.get("beam_core", {})
    display({
        "full_map_E_nonaxisymmetric_RMS_percent": 100.0 * e_nonaxis,
        "full_map_B_nonaxisymmetric_RMS_percent": 100.0 * b_nonaxis,
        "beam_core_E_nonaxisymmetric_RMS_percent": 100.0 * float(_core_symmetry.get("combined_e_nonaxisymmetric_fraction", np.nan)),
        "beam_core_B_nonaxisymmetric_RMS_percent": 100.0 * float(_core_symmetry.get("combined_b_nonaxisymmetric_fraction", np.nan)),
        "B_is_less_axisymmetric_than_E": b_nonaxis > e_nonaxis,
        "production_representation": "cylindrical vector m=0 (forced model choice)",
    })
    field_plots.plot_symmetry_mode_spectra(symmetry_diagnostics, show=True)
else:
    print("Symmetry spectra will appear after reduction or artifact loading.")

## 5. Assess the field and the missing downstream interval

The common measured support ends at 32.9488 mm; tracking extends to 40.589 mm. The shared assessment applies the declared fill treatment first, then evaluates Maxwell, axis, cathode-origin and tail checks on the same amplitude reference.

The endpoint fields and missing voltage below come from this assessment or the loaded artifact. The modal extension remains explicitly modeled; a matching zero-tail control checks its effect. Numerical qualification does not establish absolute steady-state calibration.


In [ ]:
selected_field = None
zero_control_field = None
quality_record = dict(loaded_artifact.quality) if loaded_artifact is not None else {}

if reduction_result is not None:
    assessment = fm.assess_reduced_field(
        reduction_result, grid_plan,
        simulated_duration_s=float(manifest.root_attributes["simulation_duration_s"]),
        cathode_origin_report=cathode_origin_report,
        source_integrity_gate={
            "passed": bool(source_report.valid and (VERIFY_SOURCE_SHA256 or SOURCE_DIGEST_PREVIOUSLY_CONFIRMED)),
            "metric": "schema valid and source digest verified now or explicitly confirmed previously",
            "verification_status": "verified_against_supplier_manifest" if VERIFY_SOURCE_SHA256 else "declared_not_recomputed",
            "previously_confirmed": SOURCE_DIGEST_PREVIOUSLY_CONFIRMED,
            "gate_version": fm.PRODUCTION_QUALITY_GATE_VERSION,
            "name": "source_integrity",
        },
        fill_policy=FILL_POLICY,
        drive_start_time_s=None if FILL_START_TIME_NS is None else FILL_START_TIME_NS * 1e-9,
        tail_policy=TAIL_POLICY,
        tail_fit_start_m=TAIL_FIT_START_M,
        wall_guard_cells=reduction_config.wall_guard_cells,
        axis_guard_cells=1,
        beam_core_radius_m=reduction_config.beam_core_radius_m,
    )
    measured_field = assessment.measured_field
    tail_qualification = assessment.tail
    quality_record = assessment.quality
    zero_control_field = fm.apply_tail_policy(measured_field, grid_plan, tail_qualification, policy="zero")
    if TAIL_POLICY != "modal" or tail_qualification.passed:
        selected_field = fm.apply_tail_policy(measured_field, grid_plan, tail_qualification, policy=TAIL_POLICY)
    else:
        print("Modal tail refused:", quality_record["failed_gates"])
elif loaded_artifact is not None:
    selected_field = loaded_artifact.field

if quality_record:
    numerical = quality_record.get("numerical_measured_support", {})
    gates = quality_record.get("gates", {})

    # Show the verdict as a gate-by-gate table first. Dumping every residual next to
    # "status: qualified" invites the reader to think the large full-aperture numbers below
    # were the ones judged -- they are not, and saying so explicitly is the whole point.
    display(Markdown("**Production verdict** (only these decide qualification)"))
    display({
        name: {
            "value": gate.get("value"),
            "limit": gate.get("maximum", gate.get("minimum")),
            "passed": gate.get("passed"),
            "metric": gate.get("metric", ""),
        }
        for name, gate in sorted(gates.items())
    })
    _failed = quality_record.get("failed_gates", [])
    print(f"status: {quality_record.get('status')} | failed gates: {_failed or 'none'}")

    display(Markdown(
        "**Gated region — beam core.** Maxwell residuals are qualified inside the declared "
        "beam-core cylinder, away from the axis singularity and the wall/material edges."
    ))
    display(numerical.get("maxwell_beam_core", {}).get("active_guarded_vacuum", {}))

    display(Markdown(
        "**Diagnostic only — full aperture, NOT gated.** These cover the whole guarded vacuum "
        "aperture, including a narrow off-axis locus near z~2 mm, r~7-9 mm where "
        "non-axisymmetric coupler/material structure is absent from the cylindrical m=0 "
        "model. Their large relative values are expected there and are reported, not hidden; "
        "they are deliberately excluded from the production decision above."
    ))
    display(numerical.get("maxwell_vacuum", {}).get("active_guarded_vacuum", {}))

    display(Markdown("**Axis regularity, fitted tail, and cathode origin**"))
    display({
        "axis_regularity": numerical.get("axis_regularity", {}),
        "tail_qualification": quality_record.get("tail_extension", {}),
        "cathode_origin": quality_record.get("cathode_origin", {}),
        "fill_transient": quality_record.get("fill_transient", {}),
    })
else:
    print("Maxwell, axis, and fitted-tail diagnostics will appear after a reduction or artifact load.")


In [ ]:
# Endpoint evidence on the same amplitude reference as the selected map.
_tail_record = quality_record.get("tail_extension", {})
_m = _tail_record.get("metrics", {})
if _m:
    completed_m0_endpoint_check = {
        "source_support_end_mm": 1.0e3 * float(_m["measured_z_max_m"]),
        "tracking_end_mm": 1.0e3 * float(_m["target_z_max_m"]),
        "missing_length_mm": 1.0e3 * float(_m["missing_length_m"]),
        "max_E_end_to_peak_percent": 1.0e2 * float(_m["endpoint_e_fraction_of_map_peak"]),
        "max_B_end_to_peak_percent": 1.0e2 * float(_m["endpoint_b_fraction_of_map_peak"]),
        "energy_per_length_end_to_peak": float(_m["endpoint_energy_per_length_fraction_of_peak"]),
        "fitted_decay_length_mm": 1.0e3 * float(_m["fitted_decay_length_m"]),
        "analytic_TM01_decay_length_mm": 1.0e3 * float(_m["analytic_tm01_decay_length_m"]),
        "log_amplitude_R_squared": float(_m["log_amplitude_r_squared"]),
        "maximum_fit_phase_change_deg": float(_m["fit_phase_change_deg"]),
        "measured_axis_voltage_kV": float(_m["measured_complex_voltage_magnitude_V"]) / 1.0e3,
        "modal_missing_voltage_kV": float(_m["modal_tail_integrated_voltage_V"]) / 1.0e3,
        "modal_missing_voltage_fraction_percent": 1.0e2 * float(_m["modal_tail_voltage_fraction"]),
    }
    display(completed_m0_endpoint_check)
    _failed_tail = [k for k, v in _tail_record.get("checks", {}).items() if not v]
    print("tail checks:", "all passed" if not _failed_tail else f"FAILED {_failed_tail}")
else:
    completed_m0_endpoint_check = {}
    print("No tail record available yet: run the reduction or load the artifact first.")


In [ ]:
# Modal vs zero-tail A/B. On a reuse run the control comes from its own artifact on disk,
# so this comparison is never silently skipped just because the reduction did not re-run.
_modal_field = selected_field
_zero_field = zero_control_field
_zero_source = "recomputed in this session"
if _zero_field is None and ZERO_CONTROL.exists():
    _zero_artifact = fm.read_axisymmetric_artifact(
        ZERO_CONTROL, verify="payload", require_qualified=False
    )
    if _zero_artifact.quality.get("variant") != "zero_tail_control":
        raise ValueError(
            f"{ZERO_CONTROL} is not the zero-tail control "
            f"(variant={_zero_artifact.quality.get('variant')!r})"
        )
    _zero_field = _zero_artifact.field
    _zero_source = f"loaded from {ZERO_CONTROL}"

if quality_record.get("variant") != "modal":
    print("Select a modal map to compare it with the zero-tail control.")
elif _modal_field is not None and _zero_field is not None:
    display({
        "tail_policy_selected": quality_record["variant"],
        "zero_control_source": _zero_source,
        **compare_tail_control(
            _modal_field, _zero_field,
            measured_z_max_m=quality_record["tail_extension"]["metrics"]["measured_z_max_m"],
        ),
    })
else:
    print("Modal/zero comparison needs both the selected field and the zero-tail control.")


## 6. Write once, then verify the compact artifact

The writer is atomic. The reload checks metadata and every numeric payload digest. A source or processing change must produce a different artifact identity. `quality.status='qualified'` is required by production loading; exploratory artifacts remain `analysis_only`.

In [ ]:
written_payload_sha256 = None
if WRITE_ARTIFACT:
    if reduction_result is None or selected_field is None:
        raise RuntimeError("A successful reduction and selected tail are required before writing")
    provenance = {
        "source_path": str(SOURCE.resolve()),
        "source_sha256": EXPECTED_SOURCE_SHA256,
        "source_file_size_bytes": manifest.file_size_bytes,
        "source_digest_previously_confirmed": SOURCE_DIGEST_PREVIOUSLY_CONFIRMED,
        "processing_config": {
            "grid_plan": grid_plan.as_metadata(),
            "reduction": reduction_config.as_metadata(),
            "tail_policy": TAIL_POLICY,
            "fill_treatment": quality_record["fill_transient"],
            "tail_fit_start_m": TAIL_FIT_START_M,
            "quality_gate_version": fm.PRODUCTION_QUALITY_GATE_VERSION,
        },
        "source_manifest": {
            "frequency_hz": float(manifest.drive_frequency_hz),
            "source_power_w": float(manifest.source_power_w),
            "cathode_origin_native_m": manifest.cathode_origin_native_m.tolist(),
        },
    }
    written_payload_sha256 = fm.write_axisymmetric_artifact(
        ARTIFACT,
        selected_field,
        transform=fm.XFD_TD_CATHODE_TO_BEAM,
        quality=quality_record,
        provenance=provenance,
        vacuum_mask=(
            grid_plan.measured_vacuum_mask if TAIL_POLICY == "none" else grid_plan.full_vacuum_mask
        ),
        aperture_radius_m=(
            grid_plan.measured_aperture_radius_m
            if TAIL_POLICY == "none" else grid_plan.full_aperture_radius_m
        ),
        overwrite=OVERWRITE_ARTIFACT,
    )
    print(f"wrote {ARTIFACT}")
    if WRITE_ZERO_CONTROL:
        zero_quality = dict(quality_record, variant="zero_tail_control")
        zero_provenance = dict(provenance)
        zero_provenance["processing_config"] = dict(provenance["processing_config"], tail_policy="zero")
        fm.write_axisymmetric_artifact(
            ZERO_CONTROL, zero_control_field, transform=fm.XFD_TD_CATHODE_TO_BEAM,
            quality=zero_quality, provenance=zero_provenance,
            vacuum_mask=grid_plan.full_vacuum_mask,
            aperture_radius_m=grid_plan.full_aperture_radius_m,
            overwrite=OVERWRITE_ARTIFACT,
        )
        fm.read_axisymmetric_artifact(ZERO_CONTROL, verify="payload", require_qualified=False)
        print(f"wrote and verified {ZERO_CONTROL}")

verified_artifact = None
if written_payload_sha256 is not None:
    verified_artifact = fm.read_axisymmetric_artifact(
        ARTIFACT, verify="payload", require_qualified=False
    )
    if written_payload_sha256 is not None:
        assert verified_artifact.payload_sha256 == written_payload_sha256
    display({
        "artifact": str(ARTIFACT),
        "payload_sha256": verified_artifact.payload_sha256,
        "status": verified_artifact.quality.get("status"),
        "shape_(Nz,Nr)": verified_artifact.field.shape,
        "frequency_GHz": verified_artifact.field.frequency_hz / 1.0e9,
    })
    if verified_artifact.quality.get("status") == "qualified":
        fm.read_axisymmetric_artifact(ARTIFACT, verify="payload", require_qualified=True)
        print("production qualification gate passed")


## 7. Selected meridional cross section and measured-versus-modal support

The four panels below use one common RF phase and the correct cylindrical parity across signed radius. They are the processed cross-section maps selected for RF-Track: $E_r$, $E_z$, $B_\theta$, and $B_z$. Hatching identifies the modelled interval beyond the measured HDF5 support. No legacy MAT map or solver-axis panel is involved.

In [ ]:
artifact_for_plot = verified_artifact if verified_artifact is not None else loaded_artifact
field_for_plot = artifact_for_plot.field if artifact_for_plot is not None else selected_field
quality_for_plot = (dict(artifact_for_plot.quality) if artifact_for_plot is not None else quality_record)

if field_for_plot is not None:
    if artifact_for_plot is not None and artifact_for_plot.vacuum_mask is not None:
        plot_mask = artifact_for_plot.vacuum_mask
        plot_aperture = artifact_for_plot.aperture_radius_m
    else:
        plot_mask = grid_plan.measured_vacuum_mask if TAIL_POLICY == "none" else grid_plan.full_vacuum_mask
        plot_aperture = grid_plan.measured_aperture_radius_m if TAIL_POLICY == "none" else grid_plan.full_aperture_radius_m
    measured_limit_m = float(quality_for_plot.get("tail_extension", {}).get("metrics", {}).get(
        "measured_z_max_m", grid_plan.measured_z_max_m
    ))
    measured_plot_mask = plot_mask & (field_for_plot.z_m[:, None] <= measured_limit_m + 1.0e-15)
    field_plots.plot_meridional_fields(
        field_for_plot, phase_deg=0.0, vacuum_mask=plot_mask,
        measured_mask=measured_plot_mask, measured_z_max_m=measured_limit_m,
        aperture_radius_m=plot_aperture,
        title="EM FDTD Solver (XFdtd): fields selected for RF-Track",
        show=True,
    )
    selected_numerical = quality_for_plot.get("numerical_measured_support", {})
    tail_plot_diagnostics = selected_numerical.get("tail")
    field_plots.plot_on_axis_tail_decay(
        field_for_plot, vacuum_mask=plot_mask,
        tail_diagnostics=tail_plot_diagnostics,
        tail_start_z_m=TAIL_FIT_START_M, measured_z_max_m=measured_limit_m,
        show=True,
    )
else:
    print("No processed field is available to plot. Build or copy the compact artifact first.")


## Operational hand-off

For routine builds, use `prepare_xfdtd_fieldmap.py`; it applies the same bounded workflow and writes `field_maps/cavity_axisymmetric_m0_rftrack.h5`. The tracking notebook and SLURM automation should load only that digest-verified artifact. They receive the already fitted complex fields, grid spacings, drive frequency, source-power normalization, measured/extrapolated support, aperture mask, and quality/provenance record—there is no field-map recomputation in a tracking run.

The standard simulation uses the stored axisymmetric E and B fields. A magnetic-field sensitivity control must explicitly zero `Btheta` and `Bz`; a downstream-support sensitivity control must explicitly select the zero-tail artifact or policy. The 50 um radial rerun is used to establish mesh convergence against this 100 um radial / 50 um longitudinal baseline. See the README section on field maps for the artifact contract.